# Reproduction notebook

**Topology optimized mechanical metamaterials are algorithmically simple**

Md. Hasibur Rahman Hamim et al. Deposited data and code: `10.5281/zenodo.21361351`.

---

Run this notebook top to bottom. Each section prints a number, then states the value the
manuscript reports. If the two match, that part of the paper reproduces on your machine.

- **§1** environment
- **§2** load helpers from the analysis notebook
- **§3** solver verification (4 checks, seconds)
- **§4** inspect the primary checkpoint
- **§5** δ = −1.00 with margins  ← Figure 1, §3.1
- **§6** heavy-tailed neutral-set distribution  ← Figure 4, §3.3
- **§7** robustness across every swept condition  ← Figure 5, §3.4
- **§8** optional: regenerate one shear optimum from a random seed

The file needs `Improved_Code_new.ipynb` in the same directory and `sweep_data/*.pkl` in
`./sweep_data/`. Both are in the deposit.

Total runtime with the deposited data: about a minute. §8 is optional and takes several
minutes, so it is off by default.

## 1. Environment

In [ ]:
import sys, platform, importlib

print(f"python {sys.version.split()[0]}  on  {platform.system()} {platform.machine()}\n")
for pkg in ("numpy", "scipy", "matplotlib", "joblib", "pybdm"):
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg:12s} {getattr(m, '__version__', 'unknown')}")
    except ImportError:
        note = "   <- BDM unavailable; §5 will report CLZ and zlib only" if pkg == "pybdm" else ""
        print(f"  {pkg:12s} NOT INSTALLED{note}")

## 2. Load helpers from the analysis notebook

The cell below pulls definitions out of `Improved_Code_new.ipynb` so this file cannot
drift from the code that produced the paper. It executes only cells that contain
definitions — imports, functions, classes, assignments — so no sweep runs by accident.

Two helpers used further down (`tkey`, `cliffs`) live in figure cells that the loader
correctly skips, so we define them here exactly as in the analysis code.

In [ ]:
import ast, glob, json, os, pickle
import numpy as np

# ---- locate the analysis notebook ------------------------------------------------
_nb = [f for f in (glob.glob("Improved_Code_new*.ipynb") or glob.glob("*.ipynb"))
       if "reproduction" not in f.lower() and "documentation" not in f.lower()]
if not _nb:
    raise RuntimeError("Improved_Code_new.ipynb not found next to this notebook")
NB = _nb[0]

# ---- load definition cells only --------------------------------------------------
_SAFE = (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef,
         ast.ClassDef, ast.Assign, ast.AnnAssign)
_OK_CALLS = {"filterwarnings", "simplefilter", "seterr", "set_printoptions",
             "use", "update", "makedirs", "print", "setrecursionlimit"}

def _defs_only(node):
    if isinstance(node, _SAFE):
        return True
    if isinstance(node, ast.Expr):
        v = node.value
        if isinstance(v, ast.Constant):
            return True
        if isinstance(v, ast.Call):
            f = v.func
            name = f.attr if isinstance(f, ast.Attribute) else getattr(f, "id", "")
            return name in _OK_CALLS
    return False

loaded = skipped = 0
for i, cell in enumerate(json.load(open(NB, encoding="utf-8"))["cells"]):
    if cell["cell_type"] != "code":
        continue
    src = "".join(cell["source"])
    if not src.strip() or src.lstrip().startswith(("%", "!")):
        continue
    try:
        tree = ast.parse(src)
    except SyntaxError:
        skipped += 1; continue
    if not all(_defs_only(n) for n in tree.body):
        skipped += 1; continue
    try:
        exec(compile(src, f"<{NB} cell {i}>", "exec"), globals())
        loaded += 1
    except Exception as e:
        print(f"  cell {i} raised on load: {type(e).__name__}"); skipped += 1

print(f"loaded {loaded} definition cells from {NB}, skipped {skipped}")
required = ["K", "Homogenizer", "topopt_hardened", "_distinct", "canonical", "key_of",
            "score_all", "filtered_random_cells", "random_pool"]
missing = [n for n in required if n not in globals()]
if missing:
    raise RuntimeError(f"required helpers missing after load: {missing}")
print("required helpers present")

In [ ]:
# ---- inline the two helpers that live only in figure cells -----------------------
# tkey groups tensors by equality within a tolerance; identical to the analysis code.
def tkey(C, tol=0.01):
    v = np.array([C[0, 0], C[1, 1], C[0, 1], C[2, 2]], float)
    return tuple((v / tol).round().astype(int))

# Cliff's delta, optima-first: delta < 0 means the optima are simpler. Ties enter the
# denominator but neither probability, so delta = -1 requires strict inequality on
# every pair.
def cliffs(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) == 0 or len(b) == 0:
        return np.nan
    gt = (a[:, None] > b[None, :]).sum()
    lt = (a[:, None] < b[None, :]).sum()
    return (gt - lt) / (len(a) * len(b))

# probe each estimator; drop any that fails to import at runtime
_probe = np.zeros((8, 8), dtype=np.uint8); _probe[2:6, 2:6] = 1
MEASURES = []
for m in ("clz", "zlib", "bdm"):
    try:
        K(_probe, m); MEASURES.append(m)
    except Exception as e:
        print(f"  estimator '{m}' unavailable ({type(e).__name__}), omitted")
print("estimators in use:", ", ".join(MEASURES))

## 3. Solver verification

Four checks on the homogenization routine. All complete in seconds.

In [ ]:
H = Homogenizer(32, nu=0.3, penal=3)

# 1. fully solid cell returns the plane-stress base tensor
solid = np.ones((32, 32))
Cs = H.solve(solid)
nu = 0.3
exact = np.array([[1/(1-nu**2), nu/(1-nu**2), 0],
                  [nu/(1-nu**2), 1/(1-nu**2), 0],
                  [0, 0, (1-nu)/(2*(1-nu**2))]])
e1 = np.abs(Cs - exact).max()

# 2. Voigt symmetry
e2 = np.abs(Cs - Cs.T).max()

# 3. translation invariance under periodic BCs
rng = np.random.default_rng(0)
x = (rng.random((32, 32)) < 0.4).astype(float)
C0 = H.solve(x)
C1 = H.solve(np.roll(np.roll(x, 7, axis=0), 11, axis=1))
e3 = np.abs(C0 - C1).max()

# 4. 90-degree rotation swaps C11 and C22
C2 = H.solve(np.rot90(x))
e4 = max(abs(C2[0,0] - C0[1,1]), abs(C2[1,1] - C0[0,0]))

print(f"  {'check':<32s}  {'error':>10s}   verdict")
print(f"  {'-'*32}  {'-'*10}   {'-'*8}")
for name, e, tol in (("solid cell vs analytical",         e1, 1e-10),
                     ("Voigt symmetry",                   e2, 1e-10),
                     ("translation invariance",           e3, 1e-10),
                     ("90-deg rotation swaps C11/C22",    e4, 1e-10)):
    print(f"  {name:<32s}  {e:10.2e}   {'OK' if e < tol else 'FAIL'}")

## 4. Inspect the primary checkpoint

`data_rmin_1.5.pkl` is the file behind every figure in the manuscript unless the caption
says otherwise.

In [ ]:
SWEEP = "sweep_data"
if not os.path.isdir(SWEEP):
    raise RuntimeError(f"'{SWEEP}/' not found; unzip the deposited data alongside this notebook")

def load(name):
    with open(os.path.join(SWEEP, name), "rb") as f:
        return pickle.load(f)

D = load("data_rmin_1.5.pkl")

def population(d, obj):
    """The population every reported delta is computed over: converged runs, then
    one representative per topology class."""
    runs = d["designs"][obj]
    conv = [x for x in runs if x.get("conv")]
    return runs, conv, _distinct(conv)

print(f"file: sweep_data/data_rmin_1.5.pkl")
print(f"  background        : {len(D.get('background', []))} filtered-random cells")
print(f"  candidate_pool    : {len(D.get('candidate_pool', []))} independent random cells\n")
print(f"  {'objective':>12s} {'runs':>6s} {'conv':>6s} {'distinct':>9s}")
for obj, label in (("shear", "shear"), ("dirx", "directional")):
    runs, conv, opt = population(D, obj)
    print(f"  {label:>12s} {len(runs):6d} {len(conv):6d} {len(opt):9d}")

## 5. Recompute the headline result: δ = −1.00

The manuscript reports Cliff's δ = −1.00 for both objectives under all three estimators.
δ = −1 is equivalent to a single inequality,

$$\max K(\text{optima}) < \min K(\text{random})$$

so this check is exact rather than statistical. The table below prints that inequality
directly along with the margin, which δ does not show — δ saturates at −1 the moment
separation is complete, however narrow the gap.

In [ ]:
def Karr(cells, m):
    return np.array([K(c["xb"], m) for c in cells], float)

fmt = f"{'objective':>12s} {'estimator':>9s} {'max K(opt)':>12s} {'min K(rand)':>13s} {'margin':>9s} {'delta':>7s}"
print(fmt); print("-" * len(fmt))
for obj, label in (("shear", "shear"), ("dirx", "directional")):
    _, _, opt = population(D, obj)
    bg = D["background"]
    for m in MEASURES:
        ko, kr = Karr(opt, m), Karr(bg, m)
        margin = kr.min() - ko.max()
        d = cliffs(ko, kr)
        print(f"{label:>12s} {m:>9s} {ko.max():12.2f} {kr.min():13.2f} {margin:+9.2f} {d:+7.2f}")
print("\nA positive margin means complete separation. Manuscript reports delta = -1.00 in every row.")

## 6. Recompute the mechanism: heavy-tailed neutral-set distribution

Group every cell in the pool by its effective tensor within a tolerance of 0.01, then
count distinct topologies in each group. The manuscript reports that most tensors are
made by one or two topologies, while the reachable optima are made by many.

In [ ]:
def neutral_sets(d, tol=0.01):
    pool = (list(d["designs"]["shear"]) + list(d["designs"]["dirx"])
            + list(d.get("background", [])) + list(d.get("candidate_pool", [])))
    groups = {}
    for c in pool:
        groups.setdefault(tkey(c["C"], tol), []).append(c)
    return {k: len(_distinct(v)) for k, v in groups.items()}, len(pool)

ns, npool = neutral_sets(D)
sizes = np.array(sorted(ns.values()))
print(f"pool of {npool} cells collapses to {len(ns)} distinct effective tensors\n")
print(f"  neutral-set size: min {sizes.min()}, median {int(np.median(sizes))}, max {sizes.max()}")

from collections import Counter
c = Counter(sizes)
print(f"  tensors realized by 1 topology  : {c[1]:5d}  ({100*c[1]/len(sizes):.0f}%)")
print(f"  tensors realized by 2 topologies: {c[2]:5d}  ({100*c[2]/len(sizes):.0f}%)")
print(f"  largest neutral set             : {sizes.max()}")

# and where the optima sit in that distribution
def neutral_set_at(d, obj, tol=0.01):
    _, _, opt = population(d, obj)
    pool = (list(d["designs"]["shear"]) + list(d["designs"]["dirx"])
            + list(d.get("background", [])) + list(d.get("candidate_pool", [])))
    groups = {}
    for c in pool:
        groups.setdefault(tkey(c["C"], tol), []).append(c)
    return len(_distinct(groups.get(tkey(opt[0]["C"], tol), []))), opt[0]["C"]

print()
for obj, label in (("shear", "shear"), ("dirx", "directional")):
    n, Copt = neutral_set_at(D, obj)
    print(f"  {label:>12s} optimum (C11 = {Copt[0,0]:.3f}, C33 = {Copt[2,2]:.3f})"
          f"  ->  {n} distinct topologies share this tensor")

### Tolerance sensitivity

If the heavy tail were an artifact of the binning choice, its shape would change with
the tolerance. It does not.

In [ ]:
print(f"  {'tol':>6s} {'tensors':>9s} {'median':>8s} {'max':>8s}")
for t in (0.005, 0.01, 0.02, 0.05):
    s = np.array(sorted(neutral_sets(D, t)[0].values()))
    print(f"  {t:6.3f} {len(s):9d} {int(np.median(s)):8d} {s.max():8d}")
print("\nThe heavy tail is present at every tolerance.")

## 7. Robustness across every swept condition

Recompute δ for shear at every filter radius, mesh resolution and base Poisson ratio in
the deposit. Conditions where too few runs converged are reported as **not tested**
rather than given a δ from a handful of designs.

In [ ]:
sweeps = (
    ("r_min", "data_rmin_{}.pkl", [1.5, 2.0, 2.5, 3.0]),
    ("N",     "data_res_N{}.pkl", [16, 32, 64]),
    ("nu",    "data_nu_{}.pkl",   [-0.8, -0.5, -0.25, 0.0, 0.2, 0.3, 0.45]),
)

hdr = (f"{'sweep':>7s} {'level':>7s} {'runs':>6s} {'conv':>6s} {'distinct':>9s} "
       f"{'random':>7s}  " + "  ".join(f"{m.upper():>6s}" for m in MEASURES))
print(hdr); print("-" * len(hdr))

for tag, pat, vals in sweeps:
    for v in vals:
        fname = pat.format(v)
        fp = os.path.join(SWEEP, fname)
        if not os.path.exists(fp):
            print(f"{tag:>7s} {str(v):>7s}   [file not in deposit: {fname}]"); continue
        d = load(fname)
        runs, conv, opt = population(d, "shear")
        bg = d.get("background", [])
        if len(opt) < 5 or len(bg) < 5:
            ds = ["n.t."] * len(MEASURES)
        else:
            ds = [f"{cliffs(Karr(opt, m), Karr(bg, m)):+.2f}" for m in MEASURES]
        print(f"{tag:>7s} {str(v):>7s} {len(runs):6d} {len(conv):6d} {len(opt):9d} "
              f"{len(bg):7d}  " + "  ".join(f"{x:>6s}" for x in ds))
print("\nn.t. = too few converged optima to test.")

## 8. Optional: regenerate one shear optimum from a random seed

This is off by default. Set `RUN_ONE_OPTIMIZATION = True` in the cell below to run a
single optimization from seed 0. It takes a few minutes and produces one design that
should be qualitatively similar to any converged shear optimum in the deposit.

In [ ]:
RUN_ONE_OPTIMIZATION = False

if RUN_ONE_OPTIMIZATION:
    import time
    rng = np.random.default_rng(0)
    H = Homogenizer(32, nu=0.3, penal=3)
    r0 = (rng.random((32, 32)) < 0.4).astype(float)
    t0 = time.time()
    xb, C, conv, iters = topopt_hardened(
        r0, H, OBJECTIVES["shear"], vol=0.4, rmin=1.5,
        sched=(1.0, 1.5, 2.0, 2.5, 3.0), iters_per=120, tol=1e-3, move=0.15,
    )
    print(f"finished in {time.time()-t0:.1f}s  conv={conv}  iters={iters}")
    print(f"  C33 (shear stiffness) = {C[2,2]:.4f}   solid-phase base 0.3846")
    print(f"  volume fraction       = {xb.mean():.3f}   target 0.400")
    import matplotlib.pyplot as plt
    plt.figure(figsize=(2.4, 2.4))
    plt.imshow(xb, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
    plt.xticks([]); plt.yticks([]); plt.title("regenerated shear optimum", fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print("skipped; set RUN_ONE_OPTIMIZATION = True to run")

---

### Summary of the checks in this notebook

| section | reproduces | manuscript reference |
|---|---|---|
| §3 | homogenization solver is correct | §2.1, SM2 |
| §5 | δ = −1.00 for both objectives, three estimators | Figure 1, §3.1 |
| §6 | heavy-tailed neutral-set distribution | Figure 4, §3.3 |
| §7 | separation holds across filter radius, mesh, Poisson ratio | Figure 5, §3.4 |

Two things are **not** in this file because they are produced by separate scripts:

- The correlation-matched baseline (Figure 2). Run `corrmatched_figures.py` from the
  deposit against the primary checkpoint.
- The threshold sweep of Figure 3. Run the corresponding cell in `Improved_Code_new.ipynb`.

Questions about the deposit: **mahabib@iut-dhaka.edu**